In [1]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [2]:

from pathlib import Path

# Update this path if your Drive folder or filename differs.
DATASET = Path(
    "/content/drive/MyDrive/ProteinCloudProject/Dataset.csv"
)

print("Dataset found:", DATASET.exists())

if DATASET.exists():
    print(f"File size: {DATASET.stat().st_size / (1024**2):.2f} MB")
else:
    print("File not found. Check the folder and filename in Google Drive.")


Dataset found: True
File size: 376.42 MB


In [3]:

import pandas as pd

# Read only a sample first.
sample = pd.read_csv(DATASET, nrows=5)

print("Columns:", sample.columns.tolist())
display(sample.head())

print("\nExample protein sequence:")
print(sample.loc[0, "seq"])

print("\nCorresponding structure labels:")
print(sample.loc[0, "sst3"])

print("\nSequence length:", len(sample.loc[0, "seq"]))
print("Label length:", len(sample.loc[0, "sst3"]))


Columns: ['pdb_id', 'chain_code', 'seq', 'sst8', 'sst3', 'len', 'has_nonstd_aa']


,pdb_id,chain_code,seq,sst8,sst3,len,has_nonstd_aa
0,1A30,C,EDL,CBC,CEC,3,False
1,1B05,B,KCK,CBC,CEC,3,False
2,1B0H,B,KAK,CBC,CEC,3,False
3,1B1H,B,KFK,CBC,CEC,3,False
4,1B2H,B,KAK,CBC,CEC,3,False



Example protein sequence:
EDL

Corresponding structure labels:
CEC

Sequence length: 3
Label length: 3


In [4]:

import pandas as pd

rows = 0
mismatches = 0
empty_records = 0
label_counts = {}

for chunk in pd.read_csv(DATASET, chunksize=20000):
    rows += len(chunk)

    seq = chunk["seq"].fillna("").astype(str).str.strip().str.upper()
    labels = chunk["sst3"].fillna("").astype(str).str.strip().str.upper()

    mismatches += (seq.str.len() != labels.str.len()).sum()
    empty_records += ((seq == "") | (labels == "")).sum()

    for s in labels:
        for label in s:
            label_counts[label] = label_counts.get(label, 0) + 1

print("Total records:", rows)
print("Length mismatches:", mismatches)
print("Empty records:", empty_records)
print("Structure label counts:", label_counts)


Total records: 477153
Length mismatches: 0
Empty records: 0
Structure label counts: {'C': 58525533, 'E': 26166197, 'H': 43559646}


In [5]:

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

DATASET = "/content/drive/MyDrive/ProteinCloudProject/Dataset.csv"

# Sample from each chunk to avoid loading the entire CSV into RAM.
chunks = []
for chunk in pd.read_csv(
    DATASET,
    usecols=["pdb_id", "chain_code", "seq", "sst3"],
    chunksize=20000
):
    n = min(1000, len(chunk))
    chunks.append(chunk.sample(n=n, random_state=42))

data = pd.concat(chunks, ignore_index=True)

# Normalize the fields.
for col in ["pdb_id", "chain_code", "seq", "sst3"]:
    data[col] = data[col].fillna("").astype(str).str.strip()

data["seq"] = data["seq"].str.upper()
data["sst3"] = data["sst3"].str.upper()

# Remove duplicate protein-chain records.
data = data.drop_duplicates(subset=["pdb_id", "chain_code"])

# Validate records and labels.
valid = (
    (data["seq"].str.len() > 0)
    & (data["seq"].str.len() == data["sst3"].str.len())
    & data["sst3"].str.fullmatch(r"[HEC]+", na=False)
    & (data["pdb_id"] != "")
)

data = data.loc[valid].copy()

# Keep sequences within the initial model's maximum length.
MAX_LEN = 512
too_long = data["seq"].str.len() > MAX_LEN
print("Sampled records:", len(data))
print(f"Records excluded for length > {MAX_LEN}:", int(too_long.sum()))

data = data.loc[~too_long].reset_index(drop=True)

# Split by protein ID to reduce leakage between related chains
# sharing the same PDB identifier.
split1 = GroupShuffleSplit(
    n_splits=1, test_size=0.20, random_state=42
)
trainval_idx, test_idx = next(
    split1.split(data, groups=data["pdb_id"])
)

trainval = data.iloc[trainval_idx].reset_index(drop=True)
test_df = data.iloc[test_idx].reset_index(drop=True)

split2 = GroupShuffleSplit(
    n_splits=1, test_size=0.125, random_state=42
)
train_idx, val_idx = next(
    split2.split(trainval, groups=trainval["pdb_id"])
)

train_df = trainval.iloc[train_idx].reset_index(drop=True)
val_df = trainval.iloc[val_idx].reset_index(drop=True)

print("\nTraining records:", len(train_df))
print("Validation records:", len(val_df))
print("Test records:", len(test_df))

# Confirm no PDB identifiers overlap across splits.
train_ids = set(train_df["pdb_id"])
val_ids = set(val_df["pdb_id"])
test_ids = set(test_df["pdb_id"])

print("Train/validation overlap:", len(train_ids & val_ids))
print("Train/test overlap:", len(train_ids & test_ids))
print("Validation/test overlap:", len(val_ids & test_ids))

print("\nTraining label distribution:")
print(train_df["sst3"].str.split("").explode().value_counts())


Sampled records: 24000
Records excluded for length > 512: 2276

Training records: 15232
Validation records: 2152
Test records: 4340
Train/validation overlap: 0
Train/test overlap: 0
Validation/test overlap: 0

Training label distribution:
sst3
C    1512538
H    1153492
E     724312
       30464
Name: count, dtype: int64


In [6]:

import numpy as np
from collections import Counter

# Confirm the target alphabet directly, without splitting strings.
observed_labels = set()
for labels in data["sst3"]:
    observed_labels.update(labels)

print("Observed structure labels:", sorted(observed_labels))
print("Expected labels: ['C', 'E', 'H']")

assert observed_labels.issubset({"C", "E", "H"}), (
    f"Unexpected labels found: {observed_labels - {'C', 'E', 'H'}}"
)

# Explicit, reproducible encodings.
AA_ALPHABET = "ACDEFGHIKLMNPQRSTVWY"
AA_TO_ID = {aa: i + 1 for i, aa in enumerate(AA_ALPHABET)}
UNKNOWN_AA_ID = len(AA_TO_ID) + 1
PAD_ID = 0

LABEL_TO_ID = {"C": 1, "H": 2, "E": 3}
ID_TO_LABEL = {v: k for k, v in LABEL_TO_ID.items()}

MAX_LEN = 512

def encode_sequence(seq):
    return [
        AA_TO_ID.get(aa, UNKNOWN_AA_ID)
        for aa in seq
    ]

def encode_labels(labels):
    return [LABEL_TO_ID[label] for label in labels]

def make_arrays(frame):
    X = np.zeros((len(frame), MAX_LEN), dtype=np.int32)
    y = np.zeros((len(frame), MAX_LEN), dtype=np.int32)
    weights = np.zeros((len(frame), MAX_LEN), dtype=np.float32)

    for i, row in enumerate(frame.itertuples(index=False)):
        seq = row.seq
        labels = row.sst3
        n = len(seq)

        X[i, :n] = encode_sequence(seq)
        y[i, :n] = encode_labels(labels)
        weights[i, :n] = 1.0

    return X, y, weights

X_train, y_train, w_train = make_arrays(train_df)
X_val, y_val, w_val = make_arrays(val_df)
X_test, y_test, w_test = make_arrays(test_df)

print("\nArray shapes:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("\nUnknown amino-acid symbols in training:")
print(sorted({
    aa
    for seq in train_df["seq"]
    for aa in seq
    if aa not in AA_TO_ID
}))


Observed structure labels: ['C', 'E', 'H']
Expected labels: ['C', 'E', 'H']

Array shapes:
X_train: (15232, 512)
y_train: (15232, 512)
X_val: (2152, 512)
X_test: (4340, 512)

Unknown amino-acid symbols in training:
['*']


In [7]:

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
tf.keras.utils.set_random_seed(SEED)

model = keras.Sequential([
    layers.Input(shape=(MAX_LEN,), dtype="int32"),
    layers.Embedding(
        input_dim=UNKNOWN_AA_ID + 1,
        output_dim=32,
        mask_zero=True
    ),
    layers.Bidirectional(
        layers.LSTM(64, return_sequences=True)
    ),
    layers.Dropout(0.25),
    layers.Dense(32, activation="relu"),
    layers.Dense(4, activation="softmax")
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=[]
)

model.summary()

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True
    ),
    keras.callbacks.ModelCheckpoint(
        "/content/best_protein_bilstm.keras",
        monitor="val_loss",
        save_best_only=True
    )
]

history = model.fit(
    X_train,
    y_train,
    sample_weight=w_train,
    validation_data=(X_val, y_val, w_val),
    epochs=8,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 512, 32)        │           704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 512, 128)       │        49,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512, 32)        │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 512, 4)         │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 54,628 (213.39 KB)

 Trainable params: 54,628 (213.39 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/8
476/476 ━━━━━━━━━━━━━━━━━━━━ 22s 35ms/step - loss: 0.8569 - val_loss: 0.7726
Epoch 2/8
476/476 ━━━━━━━━━━━━━━━━━━━━ 17s 35ms/step - loss: 0.7734 - val_loss: 0.7265
Epoch 3/8
476/476 ━━━━━━━━━━━━━━━━━━━━ 16s 33ms/step - loss: 0.7297 - val_loss: 0.6936
Epoch 4/8
476/476 ━━━━━━━━━━━━━━━━━━━━ 16s 34ms/step - loss: 0.7057 - val_loss: 0.6805
Epoch 5/8
476/476 ━━━━━━━━━━━━━━━━━━━━ 16s 34ms/step - loss: 0.6932 - val_loss: 0.6675
Epoch 6/8
476/476 ━━━━━━━━━━━━━━━━━━━━ 16s 33ms/step - loss: 0.6834 - val_loss: 0.6589
Epoch 7/8
476/476 ━━━━━━━━━━━━━━━━━━━━ 16s 33ms/step - loss: 0.6749 - val_loss: 0.6511
Epoch 8/8
476/476 ━━━━━━━━━━━━━━━━━━━━ 16s 34ms/step - loss: 0.6674 - val_loss: 0.6449


In [8]:

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

probabilities = model.predict(X_test, batch_size=32, verbose=1)
predictions = probabilities.argmax(axis=-1)

# Exclude padding positions from all reported metrics.
mask = w_test.astype(bool)
y_true = y_test[mask]
y_pred = predictions[mask]

print("Per-residue accuracy:", round(accuracy_score(y_true, y_pred), 4))
print("Macro F1:", round(f1_score(y_true, y_pred, average="macro"), 4))

print("\nClassification report:")
print(classification_report(
    y_true,
    y_pred,
    labels=[1, 2, 3],
    target_names=["C (Coil)", "H (Helix)", "E (Sheet)"],
    zero_division=0
))

print("Confusion matrix (rows=true, columns=predicted):")
print(confusion_matrix(y_true, y_pred, labels=[1, 2, 3]))


136/136 ━━━━━━━━━━━━━━━━━━━━ 4s 28ms/step
Per-residue accuracy: 0.7163
Macro F1: 0.7018

Classification report:
              precision    recall  f1-score   support

    C (Coil)       0.72      0.74      0.73    430654
   H (Helix)       0.73      0.77      0.75    323651
   E (Sheet)       0.68      0.58      0.62    207464

    accuracy                           0.72    961769
   macro avg       0.71      0.70      0.70    961769
weighted avg       0.71      0.72      0.71    961769

Confusion matrix (rows=true, columns=predicted):
[[319524  69286  41844]
 [ 59493 249697  14461]
 [ 64608  23150 119706]]


In [9]:

import json
from pathlib import Path

OUTPUT_DIR = Path("/content/protein_model_artifacts")
OUTPUT_DIR.mkdir(exist_ok=True)

model.save(OUTPUT_DIR / "protein_bilstm.keras")

config = {
    "max_len": MAX_LEN,
    "amino_acid_to_id": AA_TO_ID,
    "unknown_amino_acid_id": UNKNOWN_AA_ID,
    "padding_id": PAD_ID,
    "label_to_id": LABEL_TO_ID,
    "id_to_label": {str(k): v for k, v in ID_TO_LABEL.items()},
    "model_type": "BiLSTM",
    "target": "three-class per-residue secondary structure"
}

with open(OUTPUT_DIR / "preprocessing.json", "w") as f:
    json.dump(config, f, indent=2)

print("Saved artifacts:")
for path in OUTPUT_DIR.iterdir():
    print(path.name, round(path.stat().st_size / 1024**2, 2), "MB")


Saved artifacts:
preprocessing.json 0.0 MB
protein_bilstm.keras 0.67 MB


In [10]:

from google.colab import files
import shutil

shutil.make_archive(
    "/content/protein_model_artifacts",
    "zip",
    root_dir="/content/protein_model_artifacts"
)

files.download("/content/protein_model_artifacts.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
import sys
import tensorflow as tf

print("Python:", sys.version)
print("TensorFlow:", tf.__version__)

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
TensorFlow: 2.21.0
